In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.down_scale = nn.Sequential(
            nn.Conv2d(3,16,4,2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16,32,4,2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32,64,4,2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        self.up_scale = nn.Sequential(
            nn.ConvTranspose2d(64,32,4,2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32,16,4,2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.ConvTranspose2d(16,3,4,2, padding=1),
            nn.Sigmoid()
        )

        self.z_term = nn.Linear(64*8*8, 32)
        self.u_term = nn.Linear(64*8*8, 32)
        self.from_latent = nn.Linear(32, 64*8*8)

    def forward(self, x):
        z = self.down_scale(x)
        flattened = z.flatten(start_dim=1)
        mu = self.z_term(flattened)
        log_var = self.u_term(flattened)
        std = torch.exp(0.5 * log_var)
        e = torch.randn_like(std)

        latent = mu + std * e

        decoded_features = self.from_latent(latent)
        decoded_features = decoded_features.view(x.size(0), 64, 8, 8)
        y_hat = self.up_scale(decoded_features)
        return y_hat, mu, log_var

In [5]:
model = VAE()

In [6]:
x = torch.randn(8, 3, 64, 64)

z = model.down_scale(x)
print(z.shape)

y_hat = model(x)
print(y_hat.shape)

torch.Size([8, 64, 8, 8])
torch.Size([8, 3, 64, 64])
